# Warehouse Liquidation Strategy

This notebook builds the oracle solution for the liquidation task.

The goal is to:
- review the inventory and merchandising memo,
- normalize Q3 sales anomalies,
- protect the Home Automation category and recent launches,
- respect dependency chains,
- and produce two liquidation lists plus a final report.


In [ ]:
%pip install matplotlib scipy numpy pandas


In [ ]:
import os as _os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display
from scipy.optimize import milp, LinearConstraint, Bounds

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# Resolve paths relative to the notebook location
WORKSPACE_DIR = Path(_os.path.dirname(_os.path.abspath("__file__")))
DATA_DIR = Path(_os.path.abspath(_os.path.join(WORKSPACE_DIR, "..", "environment", "data")))

DATA_PATH = DATA_DIR / "Q3_Inventory_Sales_Data.csv"
MEMO_PATH = DATA_DIR / "merchandising_strategy.txt"

ANCHOR_DATE = pd.Timestamp("2024-09-26")
RECENCY_CUTOFF = ANCHOR_DATE - pd.Timedelta(weeks=8)
SALES_COLUMNS = [f"Week_{i}" for i in range(1, 13)]


## Load data and inspect the raw structure


In [ ]:
with open(MEMO_PATH, "r", encoding="utf-8") as f:
    memo_text = f.read()

print(memo_text)
print("\n--- DATA PREVIEW ---")

df = pd.read_csv(DATA_PATH)
df["Launch_Date"] = pd.to_datetime(df["Launch_Date"])

print("Shape:", df.shape)
display(df.head(8))
print("\nRows by category:")
display(df["Category"].value_counts().rename_axis("Category").reset_index(name="rows"))


## Quick EDA

The most important things to check early are:
- category mix,
- launch timing,
- damaged items,
- and whether dependency-linked SKUs are present.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
summary = (
    df.assign(
        status=np.where(df["Category"].eq("Home Automation"), "Protected by category", "Other"),
        recent=np.where(df["Launch_Date"] >= RECENCY_CUTOFF, "Recent launch", "Older launch"),
    )
    .groupby(["Category", "recent"])
    .size()
    .unstack(fill_value=0)
)

summary.plot(kind="bar", ax=ax)
ax.set_title("SKU count by category and launch recency")
ax.set_xlabel("Category")
ax.set_ylabel("SKU count")
ax.legend(title="")
plt.tight_layout()
plt.show()

print("Damaged rows:", int((df["Condition"] == "Damaged").sum()))
print("Rows with dependencies:", int(df["Requires_Base_SKU"].astype(str).str.len().gt(0).sum()))


## Feature engineering

For each SKU:
- use only the weeks it was actually on market during Q3,
- detect and replace one-off sales anomalies using a z-score cutoff of 2.5,
- flag damaged SKUs,
- flag dead SKUs when the normalized depletion time exceeds two years,
- and compute effective margin and effective volume.


In [ ]:
def normalized_avg_weekly_sales(row: pd.Series) -> float:
    sales = row[SALES_COLUMNS].to_numpy(dtype=float)
    weeks_since_launch = max(1, (ANCHOR_DATE - row["Launch_Date"]).days // 7)
    weeks_active = min(12, weeks_since_launch)
    active_sales = sales[-weeks_active:]

    if len(active_sales) > 1:
        stdev = np.std(active_sales, ddof=1)
        if stdev > 0:
            z_scores = np.abs((active_sales - np.mean(active_sales)) / stdev)
            anomaly_mask = z_scores >= 2.5
            if anomaly_mask.any():
                non_anomalous = active_sales[~anomaly_mask]
                fill_value = np.median(non_anomalous) if len(non_anomalous) else np.median(active_sales)
                active_sales = np.where(anomaly_mask, fill_value, active_sales)

    return float(np.mean(active_sales))

df["avg_weekly_sales"] = df.apply(normalized_avg_weekly_sales, axis=1)
df["is_damaged"] = df["Condition"].eq("Damaged")
df["is_dead"] = (~df["is_damaged"]) & ((df["Inventory_Qty"] / df["avg_weekly_sales"]) * 7 > 730)
df["is_protected"] = (df["Category"].eq("Home Automation")) | (df["Launch_Date"] >= RECENCY_CUTOFF)

df["effective_margin"] = np.where(df["is_damaged"] | df["is_dead"], 0.0, df["Inventory_Qty"] * df["Unit_Margin"])
df["effective_volume"] = df["Inventory_Qty"] * df["Unit_Volume"]

display(
    df[[
        "SKU", "Category", "Condition", "Launch_Date", "avg_weekly_sales",
        "is_damaged", "is_dead", "is_protected"
    ]].head(12)
)


## Build the optimization model

We keep the logic lexicographic:

1. maximize the number of damaged SKUs selected,
2. maximize the number of dead SKUs selected,
3. minimize total margin density on the remaining feasible solution.

Dependencies are enforced with:
- if a base SKU is selected, every dependent SKU that requires it must also be selected.


In [ ]:
eligible_indices = df.index[~df["is_protected"]].tolist()
eligible_df = df.loc[eligible_indices].copy()

index_map = {row_idx: var_idx for var_idx, row_idx in enumerate(eligible_indices)}
dependency_edges = []

for _, row in df.iterrows():
    parent_sku = row["Requires_Base_SKU"]
    if pd.notna(parent_sku) and str(parent_sku).strip():
        parent_rows = df.index[df["SKU"] == str(parent_sku)].tolist()
        child_rows = df.index[df["SKU"] == row["SKU"]].tolist()
        if parent_rows and child_rows:
            parent_row = parent_rows[0]
            child_row = child_rows[0]
            if parent_row in index_map and child_row in index_map:
                dependency_edges.append((index_map[parent_row], index_map[child_row]))

damaged = eligible_df["is_damaged"].astype(int).to_numpy()
dead = eligible_df["is_dead"].astype(int).to_numpy()
margin = eligible_df["effective_margin"].to_numpy(dtype=float)
volume = eligible_df["effective_volume"].to_numpy(dtype=float)

print("Eligible SKUs:", len(eligible_df))
print("Dependency edges inside eligible set:", len(dependency_edges))
print("Eligible damaged SKUs:", int(damaged.sum()))
print("Eligible dead SKUs:", int(dead.sum()))


In [ ]:
def build_constraints(target_count: int, damaged_exact: int | None = None, dead_exact: int | None = None):
    rows = [np.ones(len(eligible_df))]
    lower = [target_count]
    upper = [target_count]

    if damaged_exact is not None:
        rows.append(damaged.astype(float))
        lower.append(damaged_exact)
        upper.append(damaged_exact)

    if dead_exact is not None:
        rows.append(dead.astype(float))
        lower.append(dead_exact)
        upper.append(dead_exact)

    for parent_idx, child_idx in dependency_edges:
        row = np.zeros(len(eligible_df))
        row[parent_idx] = 1.0
        row[child_idx] = -1.0
        rows.append(row)
        lower.append(-np.inf)
        upper.append(0.0)

    return LinearConstraint(np.vstack(rows), np.array(lower), np.array(upper))


def solve_binary_program(cost_vector: np.ndarray, target_count: int, damaged_exact: int | None = None, dead_exact: int | None = None):
    result = milp(
        c=cost_vector,
        constraints=build_constraints(
            target_count=target_count,
            damaged_exact=damaged_exact,
            dead_exact=dead_exact,
        ),
        integrality=np.ones(len(eligible_df), dtype=int),
        bounds=Bounds(np.zeros(len(eligible_df)), np.ones(len(eligible_df))),
    )
    if result.status != 0:
        raise RuntimeError(result.message)
    return result.x > 0.5


def maximize_damaged_then_dead(target_count: int) -> tuple[int, int]:
    damaged_mask = solve_binary_program(-damaged.astype(float), target_count)
    damaged_exact = int(damaged[damaged_mask].sum())

    dead_mask = solve_binary_program(-dead.astype(float), target_count, damaged_exact=damaged_exact)
    dead_exact = int(dead[dead_mask].sum())

    return damaged_exact, dead_exact


def solve_density_optimal_selection(target_count: int, damaged_exact: int, dead_exact: int):
    t = 0.0
    best_mask = None

    for _ in range(60):
        mask = solve_binary_program(
            margin - t * volume,
            target_count,
            damaged_exact=damaged_exact,
            dead_exact=dead_exact,
        )
        total_margin = float(margin[mask].sum())
        total_volume = float(volume[mask].sum())
        new_t = total_margin / total_volume if total_volume > 0 else 0.0
        best_mask = mask

        if abs(new_t - t) < 1e-12:
            break
        t = new_t

    return best_mask


def build_ranked_output(selection_mask: np.ndarray) -> pd.DataFrame:
    chosen = eligible_df.loc[selection_mask].copy()

    chosen["priority_tier"] = np.select(
        [chosen["is_damaged"], chosen["is_dead"]],
        ["Damaged", "Dead"],
        default="Optimized",
    )
    tier_order = {"Damaged": 0, "Dead": 1, "Optimized": 2}
    chosen["tier_sort"] = chosen["priority_tier"].map(tier_order)

    chosen = chosen.sort_values(
        by=["tier_sort", "effective_volume", "SKU"],
        ascending=[True, False, True],
    ).reset_index(drop=True)

    chosen.insert(0, "Priority_Rank", np.arange(1, len(chosen) + 1))
    return chosen[["Priority_Rank", "SKU", "Product_Name"]], chosen


## Solve the 50-item and 100-item versions

The 50-item list and the 100-item list are solved independently but with the same rules.


In [ ]:
results = {}

for target_count in (50, 100):
    damaged_exact, dead_exact = maximize_damaged_then_dead(target_count)
    selection_mask = solve_density_optimal_selection(target_count, damaged_exact, dead_exact)
    ranked_output, detailed_selection = build_ranked_output(selection_mask)

    output_path = WORKSPACE_DIR / f"liquidation_list_{target_count}.csv"
    ranked_output.to_csv(output_path, index=False)

    total_margin = float(detailed_selection["effective_margin"].sum())
    reclaimed_space = float(detailed_selection["effective_volume"].sum())
    density = total_margin / reclaimed_space if reclaimed_space > 0 else 0.0

    results[target_count] = {
        "damaged_exact": damaged_exact,
        "dead_exact": dead_exact,
        "detailed_selection": detailed_selection,
        "ranked_output": ranked_output,
        "total_margin": total_margin,
        "reclaimed_space": reclaimed_space,
        "density": density,
    }

    print(f"Target {target_count}")
    print("  damaged selected:", damaged_exact)
    print("  dead selected:", dead_exact)
    print("  density:", round(density, 6))
    print("  output rows:", len(ranked_output))
    display(ranked_output.head(10))
    print()


## Save the final report

The report stores the final numeric outputs for both list sizes.


In [ ]:
final_report = pd.DataFrame([{
    "reclaimed_space_100": results[100]["reclaimed_space"],
    "total_margin_density_100": results[100]["density"],
    "total_margin_100": results[100]["total_margin"],
    "reclaimed_space_50": results[50]["reclaimed_space"],
    "total_margin_density_50": results[50]["density"],
    "total_margin_50": results[50]["total_margin"],
}])

final_report_path = WORKSPACE_DIR / "final_report.csv"
final_report.to_csv(final_report_path, index=False)

print("Saved:")
print(" -", WORKSPACE_DIR / "liquidation_list_50.csv")
print(" -", WORKSPACE_DIR / "liquidation_list_100.csv")
print(" -", final_report_path)
display(final_report)
